<a href="https://colab.research.google.com/github/prichter12/steam-steals/blob/main/steam_steals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Steam Steals
My app helps customers decide what games to buy by determining what games have the highest values by their price.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import pandas as pd

# Look for steam_games.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "steam_games.csv")

if len(found) == 0:
    print("❌ Can't find steam_games.csv in your Drive. Upload it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    games = pd.read_csv(FOLDER + "steam_games.csv")
    print(f"✅ Loaded {len(games)} games from:", FOLDER)

Mounted at /content/drive
✅ Loaded 1000 games from: /content/drive/MyDrive/Python/


In [2]:
from google.colab import userdata

steam_key = userdata.get("STEAM_API_KEY")

import requests

def get_owned_games(steam_id):
    url = "https://api.steampowered.com/IPlayerService/GetOwnedGames/v1/"

    params = {
        "key": steam_key,
        "steamid": steam_id,
        "include_appinfo": True,
        "format": "json"
    }

    response = requests.get(url, params=params, timeout=15)
    response.raise_for_status()

    return response.json()

In [4]:
def time_value (games, n):
  paid_games = games[games["price"] > 0].copy()
  paid_games["hours_per_dollar"] = paid_games["avg_playtime_hours"] / paid_games["price"]
  best = paid_games.sort_values("hours_per_dollar", ascending=False)
  return best[["name", "price", "avg_playtime_hours", "hours_per_dollar"]].head(n)

print(time_value(games, 5))

                                name  price  avg_playtime_hours  \
220                  Killing Floor 2   2.99               546.2   
418                Age of History II   4.99               349.9   
255                   Cookie Clicker   4.99               308.7   
4    Tom Clancy's Rainbow Six® Siege   3.99               236.7   
579                    Bright Memory   7.99               346.7   

     hours_per_dollar  
220        182.675585  
418         70.120240  
255         61.863727  
4           59.323308  
579         43.391740  


In [5]:
def price_reviews(games, n):
    paid_games = games[games["price"] > 0].copy()

    paid_games["review_score"] = paid_games["pct_pos_total"].rank(pct=True) * 100

    paid_games["price_score"] = paid_games["price"].rank(pct=True, ascending=False) * 100

    paid_games["reliability_score"] = paid_games["num_reviews_total"].rank(pct=True) * 100

    paid_games["value_score"] = (
        paid_games["review_score"] * 0.50 +
        paid_games["price_score"] * 0.30 +
        paid_games["reliability_score"] * 0.20
    )

    best = paid_games.sort_values("value_score", ascending=False)

    return best[["name", "price", "pct_pos_total", "num_reviews_total", "value_score"]].head(n)

print(price_reviews(games, 5))


                  name  price  pct_pos_total  num_reviews_total  value_score
63   Vampire Survivors   4.99             98             234756    96.236979
54   People Playground   9.99             98             258453    93.001302
6             Terraria   9.99             97            1102434    91.302083
32      Lethal Company   9.99             97             387146    90.807292
107     Slay the Spire   8.49             97             152746    90.494792


In [6]:
def community_value (games, n):
  paid_games = games[games["price"] > 0].copy()
  paid_games["users_per_dollar"] = (paid_games["owners_low"] / paid_games["price"]).round(0)
  best = paid_games.sort_values("users_per_dollar", ascending=False)
  return best[["name", "price", "owners_low", "users_per_dollar"]].head(n)

print(community_value(games, 5))

                                name  price  owners_low  users_per_dollar
21                          Among Us   2.99    20000000         6688963.0
328                     Black Desert   0.99     5000000         5050505.0
4    Tom Clancy's Rainbow Six® Siege   3.99    20000000         5012531.0
8                        Garry's Mod   5.99    20000000         3338898.0
42                          Palworld  22.49    50000000         2223210.0
